# Spherinator: training an emoji model

*Author: Bernd Doser (bernd.doser@h-its.org) &middot; Date: 2026-09-11 &middot; License: [Apache-2.0](../LICENSE)*

This notebook is part 1 of 2. It walks through the training half of the
[Spherinator](https://github.com/HITS-AIN/Spherinator) pipeline on the
[Emoji Dataset](https://huggingface.co/datasets/valhalla/emoji-dataset), a
small, fast collection of emoji images:

1. **Load** the dataset with a Spherinator `DataModule`.
2. **Train** a variational autoencoder whose latent space is the surface of the
   unit sphere $S^2$ — a small convolutional encoder and an upsampling decoder.
3. **Export** the trained encoder and decoder to ONNX.

Turning the exported ONNX graphs into an explorable sky with
[HiPSter](https://github.com/HITS-AIN/HiPSter) and
[Aladin Lite](https://aladin.cds.unistra.fr/AladinLite/) is the job of the
companion notebook,
[`emojis_inference.ipynb`](emojis_inference.ipynb) — see the note at the end of
this notebook for how the two connect.

In [ ]:
# The Spherinator package. Installed by `uv sync` from pyproject.toml; the
# fallback keeps the notebook runnable in a bare Jupyter container (see
# compose.yml).
try:
    import spherinator
except ImportError:
    %pip -q install git+https://github.com/HITS-AIN/Spherinator
    import spherinator

print(f"spherinator {spherinator.__version__}")

In [ ]:
%matplotlib inline
import os

import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# The ONNX graphs exported at the end of this notebook land here.
# `emojis_inference.ipynb` does not read this folder - it downloads its own
# (fully trained) copies from Hugging Face instead - but you can point it at
# these files if you want to explore the quick demo model trained below.
OUTPUT_PATH = "output"
ONNX_PATH = os.path.join(OUTPUT_PATH, "onnx")

# Fix all RNG seeds so reruns give the same model.
pl.seed_everything(42)

## 1. The dataset

`spherinator.data.DataModule` is a thin Lightning wrapper around a
[Hugging Face dataset](https://huggingface.co/docs/datasets). It streams the
requested `columns`, converts `uint8` image data to `float32` in $[0, 1]$,
optionally runs a per-column `transform`, and splits off validation and test
sets.

The [Emoji Dataset](https://huggingface.co/datasets/valhalla/emoji-dataset)
holds 2,749 emoji, each a 256 × 256 RGB PNG paired with a short text label
(e.g. "grinning face"). That is much higher resolution than the flat-shaded
icons need, so the `image` column gets a `transform` that resizes every sample
down to 64 × 64 — small enough to train in well under two minutes on a laptop
CPU.

`return_dict=False` with a single column makes the dataloader yield bare image
tensors of shape `[batch, 3, 64, 64]`, which is exactly what the autoencoder
expects — no unpacking needed in the training loop.

In [ ]:
import torchvision.transforms.functional as TF


def resize_image(image: torch.Tensor) -> torch.Tensor:
    return TF.resize(image, [64, 64], antialias=True)


datamodule = spherinator.data.DataModule(
    path="valhalla/emoji-dataset",
    columns=[{"name": "image", "transform": resize_image}],
    return_dict=False,
    batch_size=64,
    shuffle=True,
    num_workers=4,
)

Let's look at the data before modelling it. `setup("fit")` downloads the dataset
(cached after the first call) and builds the train/validation split.

In [ ]:
datamodule.setup("fit")
images = next(iter(datamodule.train_dataloader()))
print(f"batch {tuple(images.shape)}, range [{images.min():.2f}, {images.max():.2f}]")

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for ax, image in zip(axes.flat, images):
    # clamp: antialiased resizing can overshoot [0, 1] by a hair
    ax.imshow(
        image.clamp(0, 1).permute(1, 2, 0)
    )  # [3, 64, 64] -> [64, 64, 3] for imshow
    ax.axis("off")
fig.suptitle("Emoji Dataset")
fig.tight_layout()
plt.show()

## 2. A variational autoencoder on the sphere

`spherinator.models.VariationalAutoencoder` differs from a textbook VAE in the
shape of its latent space. Instead of a Gaussian in $\mathbb{R}^d$ it uses a
[power spherical](https://arxiv.org/abs/2006.04437) distribution on the unit
sphere $S^{z_{dim}-1}$:

- The encoder maps an image to a feature vector of size `encoder_out_dim`.
- A `SphereHead` turns that vector into a **direction** `z_location`
  (L2-normalised, so it lies exactly on the sphere) and a **concentration**
  `z_scale` (how sharply peaked the distribution around that direction is).
- The KL term pulls the posterior towards the *uniform* distribution on the
  sphere, weighted by `beta`. Because the sphere is compact and has no
  distinguished origin, there is no "posterior collapse to zero" — the
  representation stays spread over the whole surface.
- The decoder maps a sampled direction back to an image.

With `z_dim=3` the latent space is the ordinary 2-sphere, which is precisely why
the result can be displayed as a sky in the inference notebook.

### A tiny CNN

Flat-shaded 64 × 64 emoji icons need far less capacity than photographic
images, so the encoder is four plain convolution blocks built from
`ConsecutiveConv2DLayer`, each `Conv2d → BatchNorm2d → ReLU → MaxPool2d(2)`.
The spatial resolution halves and the channel count doubles at every step:

| stage | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| resolution | 32² | 16² | 8² | 4² |
| channels | 16 | 32 | 64 | 128 |

A final `Flatten → LazyLinear` projects the 128 × 4 × 4 feature map onto the
128-dimensional vector the `SphereHead` reads. `LazyConv2d`/`LazyLinear` infer
their input sizes on the first forward pass, so the layer list does not have to
repeat the shapes.

The decoder mirrors this: `Linear(3, 128)` lifts the latent direction, then
`UpsamplingDecoder2D` grows a 4 × 4 seed feature map to 64 × 64 by repeated
*bilinear upsampling followed by convolution* — which, unlike transposed
convolution, does not produce checkerboard artefacts. Its `base_channels=64`
keeps the decoder small too.

Together the two nets come to well under 1 M parameters — small enough to train
in well under two minutes. The production model used by the inference notebook
trades this tiny CNN for a `HuggingFaceResNetEncoder` and many more training
epochs (see "Where to go next" below), which is a lot more expensive to
reproduce but a lot sharper to explore.

In [ ]:
# Four Conv-BN-ReLU-MaxPool blocks: 64 -> 32 -> 16 -> 8 -> 4 pixels.
encoder = spherinator.models.ConvolutionalEncoder2D(
    input_dim=[3, 64, 64],
    output_dim=128,
    cnn_layers=[
        spherinator.models.ConsecutiveConv2DLayer(
            out_channels=[channels],
            kernel_size=3,
            stride=1,
            padding=1,  # 'same' padding: only the pooling changes the resolution
            pooling=nn.MaxPool2d(2),
        )
        for channels in (16, 32, 64, 128)
    ],
)

# Latent direction (3) -> feature vector (128) -> image (3, 64, 64).
decoder = spherinator.models.Sequential(
    modules=[
        nn.Linear(in_features=3, out_features=128),
        spherinator.models.UpsamplingDecoder2D(
            input_dim=128,
            output_dim=[3, 64, 64],
            base_channels=64,
            seed_size=4,  # 4 * 2**4 = 64 -> four upsampling blocks
        ),
    ]
)

model = spherinator.models.VariationalAutoencoder(
    encoder=encoder,
    decoder=decoder,
    encoder_out_dim=128,  # must match the encoder's output_dim
    z_dim=3,  # latent space is the 2-sphere
    beta=1e-6,  # weight of the KL term; small = reconstruction dominates
    reconstruction_loss=nn.L1Loss(),  # more robust than MSE on flat colour fields
    max_scale=1e4,  # clamp the concentration for numerical stability
)


def count_parameters(module: nn.Module) -> float:
    return sum(p.numel() for p in module.parameters()) / 1e6


# LazyConv2d/LazyLinear only materialise their weights on the first forward pass,
# so run one before counting parameters.
model(torch.randn(2, 3, 64, 64))
print(f"encoder {count_parameters(encoder):.2f} M parameters")
print(f"decoder {count_parameters(decoder):.2f} M parameters")
print(f"total   {count_parameters(model):.2f} M parameters")

## 3. Training

Nothing Spherinator-specific here — the model is a `LightningModule`, so the
standard `Trainer` drives it. `precision="32"` keeps everything on plain FP32
so the notebook runs the same way on a CPU-only machine as on a GPU; drop to
`precision="16-mixed"` instead if you do have a recent NVIDIA GPU and want to
roughly halve the memory footprint and speed up the convolutions.

The dataset is tiny — 2,749 emoji, 35 training batches per epoch — so unlike a
full simulation catalogue there is no need for a "fast smoke test" subset here:
`max_epochs=10` runs a full, real training pass in well under two minutes even
on a laptop CPU.

Metrics are written to `lightning_logs/`; `train_loss_recon` is the part you
want to watch, since with `beta=1e-6` the KL term is numerically tiny.

In [ ]:
trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    precision="32",
)
trainer.fit(model, datamodule=datamodule)

### Reconstructions

The honest check of an autoencoder: put images in, compare what comes out.
`model.reconstruct()` takes the *deterministic* path — it decodes `z_location`
directly rather than a sample from the posterior — which is also what the
exported ONNX graphs will do.

A 3-dimensional latent space is a brutal bottleneck: two angles for an emoji
drawn from dozens of unrelated categories — faces, animals, flags, food,
objects. Do not expect anything like pixel-perfect copies, or even much shape:
with this little room, the model mostly learns to place similarly-coloured
emoji near each other and reconstructs their *average colour*. That is not a
failure of training — it is what an honest 2-angle summary of a wildly diverse
image set looks like, and it is still enough to make the latent sphere worth
exploring.

In [ ]:
model.eval()
images = next(iter(datamodule.val_dataloader()))[:8]
with torch.no_grad():
    reconstruction = model.reconstruct(images.to(model.device)).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for column, (original, recon) in enumerate(zip(images, reconstruction)):
    axes[0, column].imshow(original.clamp(0, 1).permute(1, 2, 0))
    axes[1, column].imshow(recon.clamp(0, 1).permute(1, 2, 0))
    for row in (0, 1):
        axes[row, column].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("reconstruction", loc="left")
fig.tight_layout()
plt.show()

### Where do the emoji land?

Before exporting the model, it is worth confirming that the encoder actually
*uses* the sphere. If training had collapsed, every emoji would sit in one
small patch and the sky generated in the inference notebook would be uniform
mush.

The plot below encodes the validation emoji and shows their latent directions
in a Mollweide projection — the same projection an all-sky map uses.

In [ ]:
latent = []
with torch.no_grad():
    for batch, _ in zip(datamodule.val_dataloader(), range(40)):
        z_location, _ = model.encode(batch.to(model.device))
        latent.append(z_location.cpu().numpy())
latent = np.concatenate(latent)

# Cartesian direction on the unit sphere -> longitude/latitude in radians.
longitude = np.arctan2(latent[:, 1], latent[:, 0])
latitude = np.arcsin(np.clip(latent[:, 2], -1.0, 1.0))

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111, projection="mollweide")
ax.scatter(longitude, latitude, s=2, alpha=0.3)
ax.grid(True)
ax.set_title(f"Latent directions of {len(latent)} emoji")
plt.show()

## 4. Export to ONNX

HiPSter, used in the inference notebook, does not import the PyTorch model: it
loads [ONNX](https://onnx.ai/) graphs through `onnxruntime`. That decoupling is
deliberate — tile generation needs no autograd, no Lightning, and no GPU, so it
can run wherever the tiles are being published.

Two graphs are needed, and each gets a thin `nn.Module` wrapper because
`VariationalAutoencoder.forward` returns the whole training tuple
(distributions, samples, reconstruction) rather than a single tensor:

| graph | input | output | used for |
|---|---|---|---|
| `encoder.onnx` | image `x` `[N, 3, 64, 64]` | direction `[N, 3]` | placing real emoji on the sky |
| `decoder.onnx` | direction `z` `[N, 3]` | image `[N, 3, 64, 64]` | painting the HiPS tiles |

Two details matter for HiPSter:

- **The input name is the `forward` argument name.** HiPSter's `Inference` class
  passes the input under a configurable `input_name` — hence `forward(self, x)`
  for the encoder and `forward(self, z)` for the decoder.
- **The batch axis must be dynamic.** `dynamic_shapes={"z": {0: "batch"}}` marks
  it as such; without it the graph is frozen at the batch size of the example
  input, and HiPSter calls the decoder with `hierarchy²` rows at a time.

`spherinator.models.export_onnx()` does all of this from a checkpoint plus a
model YAML file, which is the right entry point in a scripted workflow. Here the
trained model is already in memory, so we call `torch.onnx.export` directly.

In [ ]:
class EncoderONNX(nn.Module):
    """Image -> latent direction, taking the deterministic mean of the posterior."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z_location, _z_scale = self.model.sphere_head(self.model.encoder(x))
        return z_location


class DecoderONNX(nn.Module):
    """Latent direction -> image."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.model.decoder(z)


os.makedirs(ONNX_PATH, exist_ok=True)
model = model.to("cpu").eval()

with torch.no_grad():
    exported = torch.onnx.export(
        EncoderONNX(model),
        (torch.randn(1, 3, 64, 64),),
        dynamic_shapes={"x": {0: "batch"}},
        dynamo=True,
        opset_version=19,
    )
    exported.optimize()
    exported.save(os.path.join(ONNX_PATH, "encoder.onnx"))

    exported = torch.onnx.export(
        DecoderONNX(model),
        (torch.randn(1, 3),),
        dynamic_shapes={"z": {0: "batch"}},
        dynamo=True,
        opset_version=19,
    )
    exported.optimize()
    exported.save(os.path.join(ONNX_PATH, "decoder.onnx"))

Verify the exported graphs: the declared signatures should show a symbolic
`batch` dimension, and running the ONNX encoder must reproduce the PyTorch
output to floating-point tolerance.

In [ ]:
try:
    import onnxruntime as ort
except ImportError:
    %pip -q install onnxruntime
    import onnxruntime as ort

for name in ("encoder.onnx", "decoder.onnx"):
    session = ort.InferenceSession(os.path.join(ONNX_PATH, name))
    inputs = [(i.name, i.shape) for i in session.get_inputs()]
    outputs = [o.shape for o in session.get_outputs()]
    print(f"{name}: {inputs} -> {outputs}")

# Same input through both runtimes -> same latent direction?
sample = next(iter(datamodule.val_dataloader()))[:16]
with torch.no_grad():
    expected = EncoderONNX(model)(sample).numpy()
session = ort.InferenceSession(os.path.join(ONNX_PATH, "encoder.onnx"))
actual = session.run(None, {"x": sample.numpy()})[0]
print(f"max |torch - onnx| = {np.abs(expected - actual).max():.2e}")

## Where to go next

The two graphs in `output/onnx/` are a complete, if quick, demo model. To turn
them into an explorable sky, continue with
[`emojis_inference.ipynb`](emojis_inference.ipynb).

That notebook downloads a much more thoroughly trained production model from
Hugging Face ([`bernddoser/emoji`](https://huggingface.co/bernddoser/emoji) —
a `HuggingFaceResNetEncoder` on the same $S^2$ latent space, trained on
128 × 128 emoji for far longer than two minutes) rather than reading
`output/onnx/`.
If you want to explore *this* notebook's quick model instead, point the
`hf_hub_download` calls there at `os.path.join(ONNX_PATH, "encoder.onnx")` /
`"decoder.onnx"` and change its image resize back to 64 × 64 to match.

Other directions worth trying here:

- **A more homogeneous subset.** The full dataset spans faces, animals, flags,
  food and objects, which is too much variety for 2 angles to capture as
  anything but colour. Pre-filter the Hugging Face dataset to emoji whose
  `text` contains e.g. `"face"` (`load_dataset(...).filter(...)`, saved locally
  and pointed to via `DataModule`'s `path`) and reconstructions get noticeably
  sharper, the same way the IllustrisTNG demo benefits from every galaxy
  sharing a common visual grammar.
- **Longer training.** 10 epochs on 2,749 emoji is already a full run; try more
  epochs or a lower learning rate and watch `train_loss_recon` in
  `lightning_logs/` keep falling.
- **A different bottleneck.** `z_dim=3` is what makes the latent space a *sky*,
  but Spherinator supports any $S^{n-1}$; higher `z_dim` reconstructs better at
  the cost of direct visualisability.
- **Stronger regularisation.** Increase `beta` to spread the emoji more
  uniformly over the sphere, or use `spherinator.callbacks.KLAnnealing` to ramp
  it up during training.
- **Other encoders.** `HuggingFaceResNetEncoder` and `HuggingFaceViTEncoder`
  swap in as drop-in replacements if you want to trade size for accuracy —
  worthwhile once the dataset is bigger than a few thousand images. This is
  what the production model on Hugging Face uses.

Further reading: Polsterer, Doser, Fehlner & Trujillo-Gomez,
[*Spherinator and HiPSter: Representation Learning for Unbiased Knowledge
Discovery from Simulations*](https://arxiv.org/abs/2406.03810) (2024), and the
[Spherinator documentation](https://spherinator.readthedocs.io).